In [1]:
import os
import sys
import numpy as np
import pydot
import rmgpy.chemkin
import rmgpy.tools.fluxdiagram
import copy

import PIL

import networkx as nx

import pickle

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append(os.environ['DATABASE_DIR'])
import database_fun

import rmgpy.data.thermo

Loading DFT database from /home/moon/autoscience_workflow/database


In [210]:
# pick time of interest
t_sample = 1e-7
t_sample = 1e-6
t_sample = 1e-5
# t_sample = 1e-4
# t_sample = 1e-3
# t_sample = 1e-2
# t_sample = 1e-1



# Load mechanism info

In [211]:
# Options controlling the individual flux diagram renderings:
program = 'dot'  # The program to use to lay out the nodes and edges
max_node_count = 50  # The maximum number of nodes to show in the diagram
max_edge_count = 50  # The maximum number of edges to show in the diagram
concentration_tol = 1e-6  # The lowest fractional concentration to show (values below this will appear as zero)
species_rate_tol = 1e-6  # The lowest fractional species rate to show (values below this will appear as zero)
max_node_pen_width = 7.0  # The thickness of the border around a node at maximum concentration
max_edge_pen_width = 9.0  # The thickness of the edge at maximum species rate


In [212]:
species_image_path = './species_images'  # where to get/place the images of each species later on
os.makedirs(species_image_path, exist_ok=True)


diagram_base_name = 'my_compare_example'
os.makedirs(diagram_base_name, exist_ok=True)

# --------------------------------- EDIT THIS ----------------------------
mech_1_label = 'RMG_min_1'
mech_1_inp = f'/home/moon/propane/RMG_min_1/chem_annotated.inp'
mech_1_dict = f'/home/moon/propane/RMG_min_1/species_dictionary.txt'

mech_2_label = 'RMG_min_2'
mech_2_inp = f'/home/moon/propane/RMG_min_2/chem_annotated.inp'
mech_2_dict = f'/home/moon/propane/RMG_min_2/species_dictionary.txt'



# Still have to load the mechanisms to get the lists of species and reactions
species_list1, _reaction_list1 = rmgpy.chemkin.load_chemkin_file(mech_1_inp, mech_1_dict)
species_list2, _reaction_list2 = rmgpy.chemkin.load_chemkin_file(mech_2_inp, mech_2_dict)

num_species1 = len(species_list1)
num_species2 = len(species_list2)


times1 = np.load(os.path.join(os.path.dirname(mech_1_inp), 'times.npy'))
concentrations1 = np.load(os.path.join(os.path.dirname(mech_1_inp), 'concentrations.npy'))
reaction_rates1 = np.load(os.path.join(os.path.dirname(mech_1_inp), 'rates.npy'))

times2 = np.load(os.path.join(os.path.dirname(mech_2_inp), 'times.npy'))
concentrations2 = np.load(os.path.join(os.path.dirname(mech_2_inp), 'concentrations.npy'))
reaction_rates2 = np.load(os.path.join(os.path.dirname(mech_2_inp), 'rates.npy'))

assert concentrations1.shape[1] == num_species1
assert concentrations2.shape[1] == num_species2

# you have to come up with your own mapping if the number of cantera reactions doesn't match the number of RMG reactions
with open(os.path.join(os.path.dirname(mech_1_inp), 'ct2rmg_rxn.pickle'), 'rb') as f:
    rxn_map1 = pickle.load(f)

# you have to come up with your own mapping if the number of cantera reactions doesn't match the number of RMG reactions
with open(os.path.join(os.path.dirname(mech_2_inp), 'ct2rmg_rxn.pickle'), 'rb') as f:
    rxn_map2 = pickle.load(f)

assert reaction_rates1.shape[1] == len(rxn_map1)
assert reaction_rates2.shape[1] == len(rxn_map2)

reaction_list1 = []
for key in rxn_map1:
    reaction_list1.append(_reaction_list1[rxn_map1[key]])
print('constructed reaction_list1')
reaction_list2 = []
for key in rxn_map2:
    reaction_list2.append(_reaction_list2[rxn_map2[key]])
print('constructed reaction_list2')


constructed reaction_list1
constructed reaction_list2


# compute big flux rates matrices

In [213]:
# Compute the rates between each pair of species (build up big matrices)
species_rates1 = np.zeros((len(times1), num_species1, num_species1), float)
for index1, reaction1 in enumerate(reaction_list1):
    rate1 = reaction_rates1[:, index1]
    if not reaction1.pairs: reaction1.generate_pairs()
    for reactant1, product1 in reaction1.pairs:
        reactant_index1 = species_list1.index(reactant1)
        product_index1 = species_list1.index(product1)
        species_rates1[:, reactant_index1, product_index1] += rate1
        species_rates1[:, product_index1, reactant_index1] -= rate1

# Compute the rates between each pair of species (build up big matrices)
species_rates2 = np.zeros((len(times2), num_species2, num_species2), float)
for index2, reaction2 in enumerate(reaction_list2):
    rate2 = reaction_rates2[:, index2]
    if not reaction2.pairs: reaction2.generate_pairs()
    for reactant2, product2 in reaction2.pairs:
        reactant_index2 = species_list2.index(reactant2)
        product_index2 = species_list2.index(product2)
        species_rates2[:, reactant_index2, product_index2] += rate2
        species_rates2[:, product_index2, reactant_index2] -= rate2


In [214]:
# Grab the closest time indices to the sample time

t_index1 = int(np.argmin(np.abs(t_sample - times1)))
t_index2 = int(np.argmin(np.abs(t_sample - times2)))

# get maximums - should this be at time of interest maybe?

In [215]:
# # Determine the maximum concentration for each species and the maximum overall concentration
# max_concentrations1 = np.max(np.abs(concentrations1), axis=0)
# max_concentration1 = np.max(max_concentrations1)

# # Determine the maximum reaction rates
# max_reaction_rates1 = np.max(np.abs(reaction_rates1), axis=0)

# # Determine the maximum rate for each species-species pair and the maximum overall species-species rate
# max_species_rates1 = np.max(np.abs(species_rates1), axis=0)
# max_species_rate1 = np.max(max_species_rates1)
# species_index1 = max_species_rates1.reshape((num_species1 * num_species1)).argsort()


# max_concentrations2 = np.max(np.abs(concentrations2), axis=0)
# max_concentration2 = np.max(max_concentrations2)

# # Determine the maximum reaction rates
# max_reaction_rates2 = np.max(np.abs(reaction_rates2), axis=0)

# # Determine the maximum rate for each species-species pair and the maximum overall species-species rate
# max_species_rates2 = np.max(np.abs(species_rates2), axis=0)
# max_species_rate2 = np.max(max_species_rates2)
# species_index2 = max_species_rates2.reshape((num_species2 * num_species2)).argsort()


# max_species_rate_total = max(max_species_rate1, max_species_rate2)
# max_concentration_total = max(max_concentration1, max_concentration2)


# -------------------------- At time of interest
# Determine the maximum concentration for each species and the maximum overall concentration
max_concentrations1 = np.max(np.abs(concentrations1[t_index1, :]))
max_concentration1 = np.max(max_concentrations1)

# Determine the maximum reaction rates
max_reaction_rates1 = np.max(np.abs(reaction_rates1[t_index1, :]))

# Determine the maximum rate for each species-species pair and the maximum overall species-species rate
max_species_rates1 = np.max(np.abs(species_rates1[t_index1, :, :]))
max_species_rate1 = np.max(max_species_rates1)
# species_index1 = max_species_rates1.reshape((num_species1 * num_species1)).argsort()


max_concentrations2 = np.max(np.abs(concentrations2[t_index2, :]))
max_concentration2 = np.max(max_concentrations2)

# Determine the maximum reaction rates
max_reaction_rates2 = np.max(np.abs(reaction_rates2[t_index2, :]))

# Determine the maximum rate for each species-species pair and the maximum overall species-species rate
max_species_rates2 = np.max(np.abs(species_rates2[t_index2, :, :]))
max_species_rate2 = np.max(max_species_rates2)
# species_index2 = max_species_rates2.reshape((num_species2 * num_species2)).argsort()


max_species_rate_total = max(max_species_rate1, max_species_rate2)
max_concentration_total = max(max_concentration1, max_concentration2)

# create mapping between models

In [216]:
species2_to_1 = {}
species2_to_1 = {key: value for key, value in zip([x for x in range(len(species_list1))], [-1] * len(species_list1))}
for i in range(len(species_list2)):
    for j in range(len(species_list1)):
        if species_list2[i].is_isomorphic(species_list1[j]):
            species2_to_1[i] = j
            break
    else:
        species2_to_1[i] = -1

species1_to_2 = {}
species1_to_2 = {key: value for key, value in zip([x for x in range(len(species_list1))], [-1] * len(species_list1))}
for i in range(len(species_list1)):
    for j in range(len(species_list2)):
        if species_list1[i].is_isomorphic(species_list2[j]):
            species1_to_2[i] = j
            break
    else:
        species1_to_2[i] = -1

# function to get the name as described by mechanism 1, use mechanism 2's name if it's not in mech1
species_names1 = [rmgpy.chemkin.get_species_identifier(sp) for sp in species_list1]
species_names2 = [rmgpy.chemkin.get_species_identifier(sp) for sp in species_list2]
def get_name_default_to_1(species_name):
    sp_index2 = species_names2.index(species_name)
    assert sp_index2 >= 0
    sp_index1 = species2_to_1[sp_index2]
    if sp_index1 < 0:
        return species_name
    return species_names1[sp_index1]
    



# Do path analysis

In [217]:
FLUX_CUTOFF = 1e-15

G1 = nx.DiGraph()
for i in range(species_rates1.shape[1]):
    for j in range(i):
        if np.abs(species_rates1[t_index1, i, j]) < FLUX_CUTOFF:
            continue

        nodeA = rmgpy.chemkin.get_species_identifier(species_list1[i])
        nodeB = rmgpy.chemkin.get_species_identifier(species_list1[j])
        if species_rates1[t_index1, i, j] > 0:  # positive flux from reactants to products, node1 to node2
            G1.add_edge(nodeA, nodeB, **{"total_flux": species_rates1[t_index1, i, j]})
        else:
            G1.add_edge(nodeB, nodeA, **{"total_flux": -species_rates1[t_index1, i, j]})


G2 = nx.DiGraph()
for i in range(species_rates2.shape[1]):
    for j in range(i):
        if np.abs(species_rates2[t_index2, i, j]) < FLUX_CUTOFF:
            continue

        nodeA = rmgpy.chemkin.get_species_identifier(species_list2[i])
        nodeB = rmgpy.chemkin.get_species_identifier(species_list2[j])

        # # Use names from mechanism 1
        # nodeA = get_name_default_to_1(rmgpy.chemkin.get_species_identifier(species_list2[i]))
        # nodeB = get_name_default_to_1(rmgpy.chemkin.get_species_identifier(species_list2[j]))
        
        if species_rates2[t_index2, i, j] > 0:  # positive flux from reactants to products, node1 to node2
            G2.add_edge(nodeA, nodeB, **{"total_flux": species_rates2[t_index2, i, j]})
        else:
            G2.add_edge(nodeB, nodeA, **{"total_flux": -species_rates2[t_index2, i, j]})


In [218]:
source_species = rmgpy.species.Species(smiles='CCC')
target_species = rmgpy.species.Species(smiles='[OH]')

def get_i_thing(thing, thing_list):
    for i in range(len(thing_list)):
        if thing.is_isomorphic(thing_list[i]):
            return i
    assert False


paths1 = list(nx.all_simple_paths(
    G1,
    source=rmgpy.chemkin.get_species_identifier(species_list1[get_i_thing(source_species, species_list1)]),
    target=rmgpy.chemkin.get_species_identifier(species_list1[get_i_thing(target_species, species_list1)]),
    cutoff=5
))

paths2 = list(nx.all_simple_paths(
    G2,
    source=rmgpy.chemkin.get_species_identifier(species_list2[get_i_thing(source_species, species_list2)]),
    target=rmgpy.chemkin.get_species_identifier(species_list2[get_i_thing(target_species, species_list2)]),
    cutoff=5
))

# paths2 = list(nx.all_simple_paths(
#     G2,
#     source=get_name_default_to_1(rmgpy.chemkin.get_species_identifier(species_list2[get_i_thing(source_species, species_list2)])),
#     target=get_name_default_to_1(rmgpy.chemkin.get_species_identifier(species_list2[get_i_thing(target_species, species_list2)])),
#     cutoff=5
# ))

In [219]:
def get_min_flux_on_path(G, path):
    path_fluxes = []
    for i in range(1, len(path)):
        data = G.get_edge_data(path[i-1], path[i])
        if data is None:
            data = G.get_edge_data(path[i], path[i-1])
            if data is None:
                return 0
            path_fluxes.append(-data['total_flux'])
        else:        
            path_fluxes.append(data['total_flux'])
    return np.min(path_fluxes)

all_path_fluxes1 = [get_min_flux_on_path(G1, p) for p in paths1]
all_path_fluxes2 = [get_min_flux_on_path(G2, p) for p in paths2]

# This report uses species names from mechanism 1 if they exist

In [220]:
CUTOFF_FRACTION = 0.99
print('================================== Mechanism 1 =======================================')

path_flux_total1 = np.sum(np.abs(all_path_fluxes1))
line_drawn1 = False
path_flux_sum1 = 0

indices1 = np.arange(len(paths1))
sorted_order1 = [x for _, x in sorted(zip(all_path_fluxes1, indices1))][::-1]

# prepare nodes and edges for skeleton graph
skeleton_nodes1 = []
skeleton_path_indices1 = []
for i in range(20):
    j = sorted_order1[i]

    if not line_drawn1:
        for k in range(len(paths1[j])):
            skeleton_nodes1.append(paths1[j][k])

    if not line_drawn1:
        skeleton_path_indices1.append(j)
    
    print(j, paths1[j], all_path_fluxes1[j])
    path_flux_sum1 += np.abs(all_path_fluxes1[j])
    if path_flux_sum1 > path_flux_total1 * CUTOFF_FRACTION and not line_drawn1:
        print('--------------------------------------------------------------------------------------')
        line_drawn1 = True

skeleton_nodes1 = list(set(skeleton_nodes1))
skeleton_paths1 = [paths1[x] for x in skeleton_path_indices1]
print()
print()
print()
print('================================== Mechanism 2 =======================================')


print_mechanism_1_names = True

path_flux_total2 = np.sum(np.abs(all_path_fluxes2))
line_drawn2 = False
path_flux_sum2 = 0

indices2 = np.arange(len(paths2))
sorted_order2 = [x for _, x in sorted(zip(all_path_fluxes2, indices2))][::-1]

# prepare nodes and edges for skeleton graph
skeleton_nodes2 = []
skeleton_path_indices2 = []
for i in range(20):
    j = sorted_order2[i]

    if not line_drawn2:
        for k in range(len(paths2[j])):
            skeleton_nodes2.append(paths2[j][k])

    if not line_drawn2:
        skeleton_path_indices2.append(j)

    if print_mechanism_1_names:
        converted_paths2 = [get_name_default_to_1(x) for x in paths2[j]]
        print(j, converted_paths2, all_path_fluxes2[j])
    else:
        print(j, paths2[j], all_path_fluxes2[j])
    path_flux_sum2 += np.abs(all_path_fluxes2[j])
    if path_flux_sum2 > path_flux_total2 * CUTOFF_FRACTION and not line_drawn2:
        print('--------------------------------------------------------------------------------------')
        line_drawn2 = True
skeleton_nodes2 = list(set(skeleton_nodes2))
skeleton_paths2 = [paths2[x] for x in skeleton_path_indices2]

================================== Mechanism 1 =======================================
1587 ['propane(1)', 'C[CH]C(22)', 'H(14)', 'O(5)', 'OH(15)'] 0.9697523598241946
554 ['propane(1)', 'C[CH2](20)', 'H(14)', 'O(5)', 'OH(15)'] 0.9697523598241946
1606 ['propane(1)', 'C[CH]C(22)', 'H(14)', 'HO2(16)', 'H2O2(17)', 'OH(15)'] 0.4580702509030079
573 ['propane(1)', 'C[CH2](20)', 'H(14)', 'HO2(16)', 'H2O2(17)', 'OH(15)'] 0.4580702509030079
1621 ['propane(1)', 'C[CH]C(22)', 'H(14)', 'H2O2(17)', 'OH(15)'] 0.40874046807263625
588 ['propane(1)', 'C[CH2](20)', 'H(14)', 'H2O2(17)', 'OH(15)'] 0.40874046807263625
2390 ['propane(1)', '[CH2]CC(23)', '[CH3](21)', 'CO[O](32)', '[CH2]OO(94)', 'OH(15)'] 0.26283625796988747
1341 ['propane(1)', '[CH3](21)', 'CO[O](32)', '[CH2]OO(94)', 'OH(15)'] 0.26283625796988747
1605 ['propane(1)', 'C[CH]C(22)', 'H(14)', 'HO2(16)', 'OH(15)'] 0.1822784735044699
572 ['propane(1)', 'C[CH2](20)', 'H(14)', 'HO2(16)', 'OH(15)'] 0.1822784735044699
1427 ['propane(1)', '[CH3](21)', '

# Construct the separate skeleton graphs in nx

In [221]:
skeleton_graph1 = nx.DiGraph()
# make the nodes first
for i in range(len(skeleton_nodes1)):
    skeleton_graph1.add_node(skeleton_nodes1[i]) 

skeleton_fluxes1 = []
for i in skeleton_path_indices1:
    for j in range(1, len(paths1[i])):
        
        nodeA = paths1[i][j - 1]
        nodeB = paths1[i][j]

        assert G1.get_edge_data(nodeA, nodeB) is not None
        
        skeleton_graph1.add_edge(nodeA, nodeB, **G1.get_edge_data(nodeA, nodeB))
        
skeleton_fluxes1 = [G1.get_edge_data(x[0], x[1])['total_flux'] for x in skeleton_graph1.edges]



skeleton_graph2 = nx.DiGraph()
# make the nodes first
for i in range(len(skeleton_nodes2)):
    skeleton_graph2.add_node(skeleton_nodes2[i]) 

skeleton_fluxes2 = []
for i in skeleton_path_indices2:
    for j in range(1, len(paths2[i])):
        
        nodeA = paths2[i][j - 1]
        nodeB = paths2[i][j]

        assert G2.get_edge_data(nodeA, nodeB) is not None
        
        skeleton_graph2.add_edge(nodeA, nodeB, **G2.get_edge_data(nodeA, nodeB))
        
skeleton_fluxes2 = [G2.get_edge_data(x[0], x[1])['total_flux'] for x in skeleton_graph2.edges]


skeleton_graph2 = nx.DiGraph()
# make the nodes first
for i in range(len(skeleton_nodes2)):
    skeleton_graph2.add_node(skeleton_nodes2[i]) 

skeleton_fluxes2 = []
for i in skeleton_path_indices2:
    for j in range(1, len(paths2[i])):
        
        nodeA = paths2[i][j - 1]
        nodeB = paths2[i][j]

        assert G2.get_edge_data(nodeA, nodeB) is not None
        
        skeleton_graph2.add_edge(nodeA, nodeB, **G2.get_edge_data(nodeA, nodeB))
        
skeleton_fluxes2 = [G2.get_edge_data(x[0], x[1])['total_flux'] for x in skeleton_graph2.edges]

# ------------------------------------------------------------------
# make a second version of skeleton2 in terms of skeleton1 names for the difference
skeleton_graph2_1 = nx.DiGraph()
# make the nodes first
for i in range(len(skeleton_nodes2)):
    skeleton_graph2_1.add_node(get_name_default_to_1(skeleton_nodes2[i]))

for i in skeleton_path_indices2:
    for j in range(1, len(paths2[i])):
        
        nodeA = paths2[i][j - 1]
        nodeB = paths2[i][j]
        assert G2.get_edge_data(nodeA, nodeB) is not None
        
        skeleton_graph2_1.add_edge(get_name_default_to_1(nodeA), get_name_default_to_1(nodeB), **G2.get_edge_data(nodeA, nodeB))

In [222]:

skeleton_edges1 = list(skeleton_graph1.edges)
skeleton_edges2 = list(skeleton_graph2.edges)

In [223]:
skeleton_difference1 = nx.difference(skeleton_graph1, skeleton_graph2_1)
print(skeleton_difference1.edges)

print()

skeleton_difference2 = nx.difference(skeleton_graph2_1, skeleton_graph1)
print(skeleton_difference2.edges)

[('H(14)', 'OH(15)')]

[('[CH2]CC(23)', 'C[CH]C(22)')]


# convert to graphviz

In [224]:
# Function to grab and generate the image for a given species
def get_image_path(species):
    species_index = str(species) + '.png'
    image_path = ''
    if not species_image_path or not os.path.exists(species_image_path):  # species_image_path is defined while loading the mechanism
        raise OSError
    for root, dirs, files in os.walk(species_image_path):
        for f in files:
            if f == species_index:
                image_path = os.path.join(root, f)
                break
    if not image_path:
        image_path = os.path.join(species_image_path, species_index)
        species.molecule[0].draw(image_path)
    return os.path.abspath(image_path)

In [225]:
# Create the graph
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
color1 = colors[0]
color2 = colors[1]

slope = -max_node_pen_width / np.log10(concentration_tol)
graph = pydot.Dot('flux_diagram', graph_type='digraph', overlap="false")
graph.set_fontname('sans')
graph.set_fontsize('10')


max_species_rate_total = max(np.max(skeleton_fluxes1), np.max(skeleton_fluxes2))
# ----------------------------ADD NODES ------------------------------#
# For Mechanism 1
for i1 in range(len(skeleton_nodes1)):
    sp_index1 = species_names1.index(skeleton_nodes1[i1])
    species1 = species_list1[sp_index1]
    node1 = pydot.Node(name=rmgpy.chemkin.get_species_identifier(species1))

    
    nodewidths = np.zeros(2)  # keep track of species concentrations/nodewidths for both mechanisms

    
    concentration1 = concentrations1[t_index1, sp_index1] / max_concentration_total
    if concentration1 < concentration_tol:
        penwidth = 0.0
    else:
        penwidth = round(slope * np.log10(concentration1) + max_node_pen_width, 3)
        nodewidths[0] = penwidth
    node1.set_penwidth(penwidth)
    node1.set_fillcolor('white')
    node1.set_color(color1)
    image_path1 = get_image_path(species1)
    if os.path.exists(image_path1):
        node1.set_image(image_path1)
        node1.set_label("")

    sp_index2 = species1_to_2[sp_index1]
    if sp_index2 >= 0:
        concentration2 = concentrations2[t_index2, sp_index2] / max_concentration_total
        if concentration2 < concentration_tol:
            penwidth = 0.0
        else:
            penwidth = round(slope * np.log10(concentration2) + max_node_pen_width, 3)
            nodewidths[1] = penwidth
            if node1.get_penwidth() > 0:
                node1.set_color('black')
            else:
                node1.set_color(color2)
                node1.set_penwidth(penwidth)


    if node1.get_color() == 'black':
        node1.set_penwidth(np.average(nodewidths))
    graph.add_node(node1)

# For Mechanism 2
for i2 in range(len(skeleton_nodes2)): 
    sp_index2 = species_names2.index(skeleton_nodes2[i2])

    species1 = species_list1[species2_to_1[sp_index2]]
    if rmgpy.chemkin.get_species_identifier(species1) in skeleton_nodes1:
        continue  # already took care of it above

    nodewidths = np.zeros(2)
    species2 = species_list2[sp_index2]
    node2 = pydot.Node(name=rmgpy.chemkin.get_species_identifier(species2))
    concentration2 = concentrations2[t_index2, sp_index2] / max_concentration_total
    if concentration2 < concentration_tol:
        penwidth = 0.0
    else:
        penwidth = round(slope * np.log10(concentration2) + max_node_pen_width, 3)
        nodewidths[1] = penwidth
    node2.set_fillcolor('white')
    node2.set_color(color2)
    node2.set_penwidth(penwidth)
    # Try to use an image instead of the label
    image_path2 = get_image_path(species2)
    if os.path.exists(image_path2):
        node2.set_image(image_path2)
        node2.set_label("")


    if node2.get_color() == 'black':
        node2.set_penwidth(np.average(nodewidths))

    graph.add_node(node2)



# ------------------------------- EDGES ------------------------------#
# Add an edge for each species-species rate
slope = -max_edge_pen_width / np.log10(species_rate_tol)

# Go through edges in Mechanism 1
for e1 in range(len(skeleton_edges1)):
    reactant_index1 = species_names1.index(skeleton_edges1[e1][0])
    product_index1 = species_names1.index(skeleton_edges1[e1][1])
    
    reactant1 = species_list1[reactant_index1]
    product1 = species_list1[product_index1]
    label1 = ''

    edge1 = pydot.Edge(rmgpy.chemkin.get_species_identifier(reactant1), rmgpy.chemkin.get_species_identifier(product1), color=color1)
    species_rate1 = species_rates1[t_index1, reactant_index1, product_index1] / max_species_rate_total
    
    if species_rate1 < 0:
        edge1.set_dir("back")
        species_rate1 = -species_rate1
    else:
        edge1.set_dir("forward")
    # Set the edge pen width
    if species_rate1 < species_rate_tol:
        penwidth = 0.0
        edge1.set_dir("none")
    else:
        penwidth = round(slope * np.log10(species_rate1) + max_edge_pen_width, 3)
    edge1.set_penwidth(penwidth)
    if label1 and label_automatically:
        edge1.set_decorate(True)
        edge1.set_label(label1)

    graph.add_edge(edge1)

# Go through edges in Mechanism 2
for e2 in range(len(skeleton_edges2)):
    reactant_index2 = species_names2.index(skeleton_edges2[e2][0])
    product_index2 = species_names2.index(skeleton_edges2[e2][1])


    # reactant already in nodes
    if rmgpy.chemkin.get_species_identifier(species_list1[species2_to_1[reactant_index2]]) in skeleton_nodes1:
        reactant2 = species_list1[species2_to_1[reactant_index2]]
    else:
        reactant2 = species_list2[reactant_index2]

    # product already in nodes
    if rmgpy.chemkin.get_species_identifier(species_list1[species2_to_1[product_index2]]) in skeleton_nodes1:
        product2 = species_list1[species2_to_1[product_index2]]
    else:
        product2 = species_list2[product_index2]
    edge2 = pydot.Edge(rmgpy.chemkin.get_species_identifier(reactant2), rmgpy.chemkin.get_species_identifier(product2), color=color2)

    species_rate2 = species_rates2[t_index2, reactant_index2, product_index2] / max_species_rate_total
    if species_rate2 < 0:
        edge2.set_dir("back")
        species_rate2 = -species_rate2
    else:
        edge2.set_dir("forward")
    # Set the edge pen width
    if species_rate2 < species_rate_tol:
        penwidth = 0.0
        edge2.set_dir("none")
    else:
        penwidth = round(slope * np.log10(species_rate2) + max_edge_pen_width, 3)



    edge2.set_penwidth(penwidth)
    graph.add_edge(edge2)


# Add Legend
graph.add_node(pydot.Node(mech_1_label + f'\nt={times1[t_index1]:.4e}', label=mech_1_label + f'\nt={times1[t_index1]:.4e}', color=color1, shape='box', penwidth=max_node_pen_width))
graph.add_node(pydot.Node(mech_2_label + f'\nt={times2[t_index2]:.4e}', label=mech_2_label + f'\nt={times2[t_index2]:.4e}', color=color2, shape='box', penwidth=max_node_pen_width))

# write in multiple formats
graph.write_png(os.path.join(diagram_base_name, f'{diagram_base_name}_{t_index1:04}.png'))




# Summarize Differences

In [239]:
def plot_kinetics(rxns, labels=None, title=None):
    """Function for plotting reaction kinetics
    Takes in a list of RMG reactions (rmgpy.reaction.Reaction) or a single reaction
    """
    plt.xlabel('1000 / T (K^-1)')
    plt.ylabel('log10(k)')
    linestyles = ['solid', 'dashed']
    if type(rxns) != list:
        rxns = [rxns]
    T = np.linspace(300, 3000, 1001)
    for m, rxn in enumerate(rxns):
        k = np.zeros(len(T))
        for i in range(0, len(T)):
            k[i] = rxn.get_rate_coefficient(T[i], 101325)
        plt.plot(1000.0 / T, np.log10(k), linestyle=linestyles[m % len(linestyles)])
    if labels:
        plt.legend(labels)
    if title:
        plt.title(title)
    plt.show()

def plot_gibbs(thermos, labels=None, title=None):
    if type(thermos) != list:
        thermos = [thermos]
    if labels is None:
        labels = ['' for t in thermos]
    linestyles = ['solid', 'solid', 'dashed', 'dashed']
    T = np.linspace(800, 900, 1001)
#     T = np.linspace(300, 3000, 1001)
    for n, thermo in enumerate(thermos):
        G = np.zeros(len(T))
        for i in range(0, len(T)):
            G[i] = thermo.get_free_energy(T[i]) / 4184
        plt.plot(T, G, linestyle=linestyles[n % len(linestyles)])
    ax = plt.gca()
    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel('G (kcal / mol)')
    ax.set_title('Gibbs Energy vs. Temperature')
    ax.legend(labels)
    if title:
        plt.title(title)
    ax.yaxis.get_major_formatter().set_useOffset(False)
    plt.subplots_adjust(wspace=0.25)
    plt.show()

In [240]:
def reactions_in_same_direction(reaction1, reaction2):
    assert reaction1.is_isomorphic(reaction2), 'Reactions are not even isomorphic'
    if len(reaction1.reactants) != len(reaction2.reactants):
        return False
    reactants_to_match = [r for r in reaction1.reactants]
    counter = 0
    while reactants_to_match:
        for i in range(len(reaction2.reactants)):
            if reaction2.reactants[i].is_isomorphic(reactants_to_match[0]):
                reactants_to_match.remove(reactants_to_match[0])
                break
        else:
            return False
        if counter >= len(reaction1.reactants):
            return False
        counter += 1
    return True

In [241]:
def edge_on_path(edge, path):
    for i in range(1, len(path)):
        if path[i - 1] == edge[0] and path[i] == edge[1]:
            return True
    return False

In [249]:
def summarize_species_differences(species1, species2, verbose=False):
    if species2 is None:
        print(f'Species {species1} not in mechanism 2')
        return
    elif species1 is None:
        print(f'Species {species2} not in mechanism 1')
        return

    try:
        db_index = database_fun.get_unique_species_index(species1)
    except IndexError:
        db_index = -1
    
    if species2.thermo.is_identical_to(species1.thermo):
        pass  # silence when identical
    elif species2.thermo.is_similar_to(species1.thermo):
        pass
        # print(f'Species {db_index} {species1} thermo are similar')
    else:
        print(f'Species {db_index} {species1} thermo are different')
        if verbose:
            print(species1.thermo)
            print(species2.thermo)
            plot_gibbs([species1, species2], labels=[mech_1_label, mech_2_label], title=str(species1))


def summarize_reaction_differences(reaction1, reaction2, verbose=False):
    if reaction2 is None:
        print(f'Reaction {reaction1} not in mechanism 2')
        return
    elif reaction1 is None:
        print(f'Reaction {reaction2} not in mechanism 1')
        return

    try:
        db_index = database_fun.get_unique_reaction_index(reaction1)
    except IndexError:
        db_index = -1
    assert reactions_in_same_direction(reaction1, reaction2)
    if reaction2.kinetics.is_identical_to(reaction1.kinetics):
        pass  # silence when identical
    elif reaction2.kinetics.is_similar_to(reaction1.kinetics):
        # print(f'Reaction {db_index} {reaction1} kinetics are similar')
        pass
    else:
        print(f'Reaction {db_index} {reaction1} kinetics are different')
        if verbose:
            print(reaction1.kinetics)
            print(reaction2.kinetics)
            plot_kinetics([reaction1, reaction2], labels=[mech_1_label, mech_2_label], title=str(reaction1))


# Things in Mech 1 that aren't in Mech 2

In [253]:
# how many paths go through the edge on skeleton1?
# for every edge get the species thermos involved
verbose = False
for j, e in enumerate(skeleton_difference1.edges):
    print(f'Analyzing edge {j}: {e}')
    print('--------------------------------------------')
    for i in range(len(skeleton_paths1)):
        if edge_on_path(e, skeleton_paths1[i]):
            
            print(skeleton_paths1[i])

            # show species differences on this path
            for sp_name in skeleton_paths1[i]:
                sp_index1 = species_names1.index(sp_name)
                species1 = species_list1[sp_index1]

                sp_index2 = species1_to_2[sp_index1]
                species2 = None
                if sp_index2 >= 0:
                    species2 = species_list2[sp_index2]
                summarize_species_differences(species1, species2, verbose=verbose)

            # show reaction differences on this path
            key_reactions1 = get_reactions_on_path1(skeleton_paths1[i])
            for k in key_reactions1:
                reaction1 = reaction_list1[k]
                try:
                    rxn_index2 = get_i_thing(reaction1, reaction_list2)
                except AssertionError:
                    rxn_index2 = -1
                    
                reaction2 = None
                if rxn_index2 >= 0:
                    reaction2 = reaction_list2[rxn_index2]
                summarize_reaction_differences(reaction1, reaction2, verbose=verbose)
            print()
            
    print()
    print()

Analyzing edge 0: ('H(14)', 'OH(15)')
--------------------------------------------
['propane(1)', '[CH2]CC(23)', '[CH3](21)', 'C[O](89)', 'H(14)', 'OH(15)']
Species 130 propane(1) thermo are different
Species 48 [CH2]CC(23) thermo are different
Reaction OH(15) + C=CC=O(1005) <=> H(14) + [O]C=CC[O](2093) not in mechanism 2
Reaction OH(15) + C=CC=O(1005) <=> H(14) + O=CCC=O(2968) not in mechanism 2

['propane(1)', '[CH2]CC(23)', 'H(14)', 'OH(15)']
Species 130 propane(1) thermo are different
Species 48 [CH2]CC(23) thermo are different
Reaction 235 H(14) + C3H6(12) <=> [CH2]CC(23) kinetics are different
Reaction OH(15) + C=CC=O(1005) <=> H(14) + [O]C=CC[O](2093) not in mechanism 2
Reaction OH(15) + C=CC=O(1005) <=> H(14) + O=CCC=O(2968) not in mechanism 2





# Things in Mech 2 that aren't in Mech 1

In [254]:
# how many paths go through the edge on skeleton2?
# for every edge get the species thermos involved
for j, e in enumerate(skeleton_difference2.edges):
    # translate the edge back into mechanism 2 naming

    sp1_index = species_names1.index(e[0])
    sp2_index = species1_to_2[sp1_index]
    assert sp2_index >= 0
    src_name = rmgpy.chemkin.get_species_identifier(species_list2[sp2_index])

    sp1_index = species_names1.index(e[1])
    sp2_index = species1_to_2[sp1_index]
    assert sp2_index >= 0
    dst_name = rmgpy.chemkin.get_species_identifier(species_list2[sp2_index])

    e = (src_name, dst_name)
    
    print(f'Analyzing edge {j}: {e}')
    print('--------------------------------------------')
    for i in range(len(skeleton_paths2)):
        if edge_on_path(e, skeleton_paths2[i]):
            
            print(skeleton_paths2[i])

            # show species differences on this path
            for sp_name in skeleton_paths2[i]:
                sp_index2 = species_names2.index(sp_name)
                species2 = species_list2[sp_index2]
                
                sp_index1 = species2_to_1[sp_index2]
                species1 = None
                if sp_index1 >= 0:
                    species1 = species_list1[sp_index1]
                summarize_species_differences(species1, species2)

            # show reaction differences on this path
            key_reactions2 = get_reactions_on_path2(skeleton_paths2[i])
            for k in key_reactions2:
                reaction2 = reaction_list2[k]
                try:
                    rxn_index1 = get_i_thing(reaction2, reaction_list1)
                except AssertionError:
                    rxn_index1 = -1
                reaction1 = None
                if rxn_index1 >= 0:
                    reaction1 = reaction_list1[rxn_index1]
                summarize_reaction_differences(reaction1, reaction2)
            print()
            
    print()
    print()

Analyzing edge 0: ('[CH2]CC(21)(23)', 'C[CH]C(28)')
--------------------------------------------
['propane(1)', '[CH2]CC(21)(23)', 'C[CH]C(28)', 'H(14)', 'H2O2(17)', 'OH(15)']
Species 130 propane(1) thermo are different
Species 48 [CH2]CC(23) thermo are different
Species 49 C[CH]C(22) thermo are different
Reaction 31699 H(14) + propane(1) <=> H2(13) + [CH2]CC(23) kinetics are different

['propane(1)', '[CH2]CC(21)(23)', 'C[CH]C(28)', 'H(14)', 'HO2(16)', 'OH(15)']
Species 130 propane(1) thermo are different
Species 48 [CH2]CC(23) thermo are different
Species 49 C[CH]C(22) thermo are different
Reaction 31699 H(14) + propane(1) <=> H2(13) + [CH2]CC(23) kinetics are different





# Get report on individual path

In [71]:
# for a given step, find the top reactions contributing to that flux pair

path_index = 1615
for path_step in range(len(paths1[path_index]) - 1):
    print(f'                                  Step {path_step + 1}')
    node1 = paths1[path_index][path_step]
    node2 = paths1[path_index][path_step + 1]
    
    print('                      ', node1, '->', node2)
    print('========================================================================')

    
    my_reactions = []
    my_rates = []
    
    for index1, reaction1 in enumerate(reaction_list1):
        rate1 = reaction_rates1[t_index1, index1]
        if not reaction1.pairs: reaction1.generate_pairs()
        for reactant1, product1 in reaction1.pairs:
            reactant_index1 = species_list1.index(reactant1)
            product_index1 = species_list1.index(product1)
    
            r = rmgpy.chemkin.get_species_identifier(species_list1[reactant_index1])
            p = rmgpy.chemkin.get_species_identifier(species_list1[product_index1])
            if r == node1 and p == node2:
                my_reactions.append(index1)
                my_rates.append(rate1)
                # print(index1, reaction_list1[index1], rate1)
            elif r == node2 and p ==node1:
                my_reactions.append(index1)
                my_rates.append(-rate1)
                # print(index1, reaction_list1[index1], -rate1)
    
    indices = np.arange(len(my_reactions))
    sorted_order = [x for _, x in sorted(zip(my_rates, indices))][::-1]
    
    CUTOFF_FRACTION = 0.9
    flux_total = np.sum(np.abs(my_rates))
    flux_sum = 0
    line_drawn = False
    for i in range(min(10, len(my_reactions))):
        
        j = sorted_order[i]
    
        reaction_index = ''
        if not line_drawn:
            reaction_index = str(database_fun.get_unique_reaction_index(reaction_list1[my_reactions[j]]))
        
        print(reaction_index, reaction_list1[my_reactions[j]], my_rates[j])
        flux_sum += np.abs(my_rates[j])
    
        if flux_sum > flux_total * CUTOFF_FRACTION and not line_drawn:
            print('--------------------------------------------------------------------')
            line_drawn = True

        if line_drawn:
            break
    print()
    print()

                                  Step 1
                       propane(1) -> C[CH]C(22)
28940 H2(13) + C[CH]C(22) <=> H(14) + propane(1) 0.12560643980783606
28941 OH(15) + propane(1) <=> H2O(8) + C[CH]C(22) 0.015586544879237321
--------------------------------------------------------------------


                                  Step 2
                       C[CH]C(22) -> H(14)
380 H(14) + C3H6(12) <=> C[CH]C(22) 0.10294670076026237
--------------------------------------------------------------------


                                  Step 3
                       H(14) -> O(5)
0 O2(2) + H(14) <=> O(5) + OH(15) 0.1028569322460113
--------------------------------------------------------------------


                                  Step 4
                       O(5) -> OH(15)
33017 O(5) + propane(1) <=> OH(15) + [CH2]CC(23) 0.02465679932328604
--------------------------------------------------------------------




In [238]:
# for a given step, find the top reactions contributing to that flux pair

def get_reactions_on_path1(path1):
    key_reactions = []
    for path_step in range(len(path1) - 1):
        # print(f'                                  Step {path_step + 1}')
        node1 = path1[path_step]
        node2 = path1[path_step + 1]
        
        # print('                      ', node1, '->', node2)
        # print('========================================================================')
    
        
        my_reactions = []
        my_rates = []
        
        for index1, reaction1 in enumerate(reaction_list1):
            rate1 = reaction_rates1[t_index1, index1]
            if not reaction1.pairs: reaction1.generate_pairs()
            for reactant1, product1 in reaction1.pairs:
                reactant_index1 = species_list1.index(reactant1)
                product_index1 = species_list1.index(product1)
        
                r = rmgpy.chemkin.get_species_identifier(species_list1[reactant_index1])
                p = rmgpy.chemkin.get_species_identifier(species_list1[product_index1])
                if r == node1 and p == node2:
                    my_reactions.append(index1)
                    my_rates.append(rate1)
                    # print(index1, reaction_list1[index1], rate1)
                elif r == node2 and p ==node1:
                    my_reactions.append(index1)
                    my_rates.append(-rate1)
                    # print(index1, reaction_list1[index1], -rate1)
        
        indices = np.arange(len(my_reactions))
        sorted_order = [x for _, x in sorted(zip(my_rates, indices))][::-1]
        
        CUTOFF_FRACTION = 0.9
        flux_total = np.sum(np.abs(my_rates))
        flux_sum = 0
        line_drawn = False
        for i in range(min(10, len(my_reactions))):
            
            j = sorted_order[i]
        
            reaction_index = ''
            if not line_drawn:
                try:
                    reaction_index = str(database_fun.get_unique_reaction_index(reaction_list1[my_reactions[j]]))
                except IndexError:
                    reaction_index = -1
                key_reactions.append(my_reactions[j])
            
            # print(reaction_index, reaction_list1[my_reactions[j]], my_rates[j])
            flux_sum += np.abs(my_rates[j])
        
            if flux_sum > flux_total * CUTOFF_FRACTION and not line_drawn:
                # print('--------------------------------------------------------------------')
                line_drawn = True
    
            if line_drawn:
                break
        # print()
        # print()
    return key_reactions


# for a given step, find the top reactions contributing to that flux pair

def get_reactions_on_path2(path2):
    key_reactions = []
    for path_step in range(len(path2) - 1):
        node1 = path2[path_step]
        node2 = path2[path_step + 1]
        
        my_reactions = []
        my_rates = []
        
        for index2, reaction2 in enumerate(reaction_list2):
            rate2 = reaction_rates2[t_index2, index2]
            if not reaction2.pairs: reaction2.generate_pairs()
            for reactant2, product2 in reaction2.pairs:
                reactant_index2 = species_list2.index(reactant2)
                product_index2 = species_list2.index(product2)
        
                r = rmgpy.chemkin.get_species_identifier(species_list2[reactant_index2])
                p = rmgpy.chemkin.get_species_identifier(species_list2[product_index2])
                if r == node1 and p == node2:
                    my_reactions.append(index2)
                    my_rates.append(rate2)
                elif r == node2 and p ==node1:
                    my_reactions.append(index2)
                    my_rates.append(-rate2)
        
        indices = np.arange(len(my_reactions))
        sorted_order = [x for _, x in sorted(zip(my_rates, indices))][::-1]
        
        CUTOFF_FRACTION = 0.9
        flux_total = np.sum(np.abs(my_rates))
        flux_sum = 0
        line_drawn = False
        for i in range(min(10, len(my_reactions))):
            
            j = sorted_order[i]
        
            reaction_index = ''
            if not line_drawn:
                try:
                    reaction_index = str(database_fun.get_unique_reaction_index(reaction_list2[my_reactions[j]]))
                except IndexError:
                    reaction_index = -1
                key_reactions.append(my_reactions[j])
            
            # print(reaction_index, reaction_list1[my_reactions[j]], my_rates[j])
            flux_sum += np.abs(my_rates[j])
        
            if flux_sum > flux_total * CUTOFF_FRACTION and not line_drawn:
                # print('--------------------------------------------------------------------')
                line_drawn = True
    
            if line_drawn:
                break
        # print()
        # print()
    return key_reactions

In [122]:
get_reactions_on_path1(paths1[1615])

[48, 47, 63, 0, 68]

In [113]:
str(reaction_list1[68])

'O(5) + propane(1) <=> OH(15) + [CH2]CC(23)'

In [ ]:
reaction_list1[48]